In [17]:
import sys
sys.path.insert(0, '..')


# Basic imports
import jax.numpy as np
import jax.random as jr
import jax.scipy as jsp
import jax
import numpy as onp
import pickle

jax.config.update("jax_enable_x64", True)


# Optimisation imports
import zodiax as zdx
import optax
import optimistix as optx
import equinox as eqx

# dLux imports
import dLux as dl
import dLux.utils as dlu

#FITS 
from astropy.io import fits
from pathlib import Path
import pandas as pd

# Visualisation imports
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
import chainconsumer as cc


%matplotlib inline
plt.rcParams['image.cmap'] = 'inferno'
plt.rcParams["font.family"] = "serif"
plt.rcParams["image.origin"] = 'lower'
plt.rcParams['figure.dpi'] = 72
plt.rcParams["font.size"] = 24

from detectors import *
from apertures import *
from models import *
from stats import posterior
from fitting import *
from plotting import *
from spectra import *

import jax.tree_util as jtu
import interpax as ipx

def set_array(pytree):
    dtype = np.float64 if jax.config.x64_enabled else np.float32
    floats, other = eqx.partition(pytree, eqx.is_inexact_array_like)
    floats = jtu.tree_map(lambda x: np.array(x, dtype=dtype), floats)
    return eqx.combine(floats, other)

def to_numpy_tree(tree):
    return jtu.tree_map(
        lambda x: onp.asarray(x)
        if hasattr(x, "shape")
        else x,
        tree
    )

In [ ]:

# Telescope / instrument starting parameters for BFGS optimization

def make_telescope_params(exposures,config):

    n_zernikes = config["n_zernikes"]
    oversample = config["oversample"]

    params = {
        "aberrations": {},

        "cold_mask_shift": {},
        "cold_mask_rot": {},
        "cold_mask_scale": {},
        "cold_mask_shear": {},

        "primary_scale": {},
        "primary_rot": {},
        "primary_shear": {},

        "outer_radius": 1.2 * 0.955,
        "secondary_radius": 0.372 * 1.2,
        "spider_width": 0.077 * 1.2,

        "scale": 0.0432,
        "softening": 2.0,
        "bias": {},
        "jitter": {},
        "defocus": {},
        "despace": {},
        "mag": 3.3,
        "quadrature": {},
    }
    
# For every exposure, initialize the telescope parameters to the starting values from the grid search results

    for exp in exposures:

        params["aberrations"][ exp.fit.get_key(exp, "aberrations") ] = np.zeros(n_zernikes)
        params["cold_mask_shift"][ exp.fit.get_key(exp, "cold_mask_shift") ] = np.asarray([8.0, 8.0])
        params["cold_mask_rot"][ exp.fit.get_key(exp, "cold_mask_rot") ] = -45.0
        params["cold_mask_scale"][ exp.fit.get_key(exp, "cold_mask_scale") ] = np.asarray([1.0, 1.0])
        params["cold_mask_shear"][ exp.fit.get_key(exp, "cold_mask_shear") ] = np.asarray([0.0, 0.0])
        params["primary_scale"][ exp.fit.get_key(exp, "primary_scale") ] = np.asarray([1.0, 1.0])
        params["primary_rot"][ exp.fit.get_key(exp, "primary_rot") ] = 45.0
        params["primary_shear"][ exp.fit.get_key(exp, "primary_shear") ] = np.asarray([0.0, 0.0])
        params["defocus"][ exp.fit.get_key(exp, "defocus") ] = 0.0
        params["despace"][ exp.fit.get_key(exp, "despace") ] = 0.0
        params["bias"][ exp.fit.get_key(exp, "bias") ] = 0.0
        params["jitter"][ exp.fit.get_key(exp, "jitter") ] = 7 / 43 * oversample
        params["quadrature"][ exp.fit.get_key(exp, "quadrature") ] = np.array(0.0)

    return ModelParams(params)

In [ ]:
def make_scales(params, scale_values):

    scales = {}

    for key in params.params:

        scale = scale_values.get( key, 1.0 )
        scales[key] = jtu.tree_map( lambda x: np.ones_like(x) * scale, params.params[key] )

    return scales

In [ ]:
def run_bfgs_for_group( target, filt, exposures_single, exposures_binary, grid_output, optics, detector, config, results_dir, make_plots=False, save=True ):

    print()
    print("=" * 60)
    print("RUNNING BFGS")
    print("TARGET:", target)
    print("FILTER:", filt)
    print("=" * 60)


    # 1. GRID STARTING PARAMETERS

    single_grid_params = ModelParams( grid_output["single_params"] )
    binary_grid_params = ModelParams( grid_output["binary_params"] )

    print("Grid results received")

    
    # 2. TELESCOPE STARTING PARAMETERS
    
    telescope_params = make_telescope_params( exposures_single, config )
    single_start_params = ( telescope_params.combine( single_grid_params ) )
    binary_start_params = ( telescope_params.combine( binary_grid_params ) )


    # 3. BUILD FULL MODELS
    
    model_single = set_array( NICMOSModel( exposures_single, single_start_params.params, optics, detector ) )
    model_binary = set_array( NICMOSModel( exposures_binary, binary_start_params.params, optics, detector ) )

    
    # 4. PARAMETERS BFGS CAN MOVE
   
    single_bfgs_keys = ( config["single_bfgs_keys"] )
    binary_bfgs_keys = ( config["binary_bfgs_keys"] )

    single_bfgs_start = ModelParams({ key: single_start_params.params[key] for key in single_bfgs_keys })
    binary_bfgs_start = ModelParams({ key: binary_start_params.params[key] for key in binary_bfgs_keys })


    # 5. PARAMETER SCALING
   
    single_scales = make_scales( single_bfgs_start, config[ "single_scale_values" ] )
    binary_scales = make_scales( binary_bfgs_start, config[ "binary_scale_values" ] )

   
    # 6. LOSS BEFORE BFGS
    
    single_loss_before = loss_fn( single_bfgs_start, exposures_single, model_single )
    binary_loss_before = loss_fn( binary_bfgs_start, exposures_binary, model_binary )

    print( "Single loss before:", single_loss_before )
    print( "Binary loss before:", binary_loss_before )


   
    # 7. SINGLE BFGS
    
    print()
    print("Running single BFGS...")

    single_bfgs_params = ( optimise_optimistix( single_bfgs_start.params, model_single, exposures_single, project=False, scales=single_scales, max_steps= config["max_steps"], progress=True, progress_desc="Single BFGS" ) )
    single_final_params = ( single_start_params.combine( single_bfgs_params ) )
    single_loss_after = loss_fn( single_bfgs_params, exposures_single, model_single )


    # 8. BINARY BFGS
   
    print()
    print("Running binary BFGS...")


    binary_bfgs_params = ( optimise_optimistix( binary_bfgs_start.params, model_binary, exposures_binary, project=False, scales=binary_scales, max_steps= config["max_steps"], progress=True, progress_desc="Binary BFGS" ) )
    binary_final_params = ( binary_start_params.combine( binary_bfgs_params ) )
    binary_loss_after = loss_fn( binary_bfgs_params, exposures_binary, model_binary )


    # 9. FINAL MODELS
   
    single_model_final = ( single_final_params.inject( model_single ) )
    binary_model_final = ( binary_final_params.inject( model_binary ) )

    
    # 10. CHECK FOR NaN / INF
    
    single_finite = True
    for exp in exposures_single:
        fit = exp.fit( single_model_final, exp )
        single_finite &= bool( np.all( np.isfinite(fit) ) )

    binary_finite = True
    for exp in exposures_binary:
        fit = exp.fit( binary_model_final, exp )
        binary_finite &= bool( np.all( np.isfinite(fit) ) )

    
    # 11. SUMMARY
    
    print()
    print("RESULTS")
    print("-" * 40)
    print( "Single:", single_loss_before, "→", single_loss_after )
    print( "Binary:", binary_loss_before, "→", binary_loss_after )
    print( "Single finite:", single_finite )
    print( "Binary finite:", binary_finite )

    
    # 12. OPTIONAL PLOTS
   
    if make_plots:

        print()
        print("Single final model")
        plot_comparison( model_single, single_final_params, exposures_single )

        print()
        print("Binary final model")
        plot_comparison( model_binary, binary_final_params, exposures_binary )


    # 13. OUTPUT
   
    bfgs_output = {

        "target": target,
        "filter": filt,
        "files": ( grid_output.get( "files", [ exp.filename for exp in exposures_single ] ) ),
        "single": { "params": to_numpy_tree( single_final_params.params ), "loss_before": onp.asarray( single_loss_before ), "loss_after": onp.asarray( single_loss_after ), "finite": single_finite, },
        "binary": { "params": to_numpy_tree( binary_final_params.params ), "loss_before": onp.asarray( binary_loss_before ), "loss_after": onp.asarray( binary_loss_after ), "finite": binary_finite, },
    }


    # 14. SAVE
    
    if save:

        safe_target = ( str(target) .replace(" ", "_") .replace("/", "_") )
        group_dir = ( Path(results_dir) / safe_target / filt )
        group_dir.mkdir( parents=True, exist_ok=True )

        save_path = ( group_dir / "bfgs.pkl" )
        with open( save_path, "wb" ) as f: pickle.dump( bfgs_output, f )
        print()
        print( "Saved:", save_path )
        bfgs_output[ "save_path" ] = str( save_path )

    return bfgs_output